# 🤖 Projeto: Previsão de Demanda de Vendas - Rossmann

## Notebook 02 - Modelagem e Comparação de Algoritmos

### 🎯 Objetivos deste Notebook
Neste notebook, vamos **construir e comparar** diferentes modelos de Machine Learning para prever vendas. É como uma competição interna onde testamos vários algoritmos para descobrir qual deles consegue fazer as melhores previsões!

### 🏆 O que vamos fazer:
1. **Treinar 4 modelos diferentes** (do mais simples ao mais sofisticado)
2. **Comparar o desempenho** de cada um usando métricas claras
3. **Entender quais variáveis** são mais importantes
4. **Visualizar os resultados** de forma profissional
5. **Escolher o campeão** e salvar para uso futuro

### 📊 Os Competidores (Modelos):

| Modelo | Tipo | Vantagens | Quando usar |
|--------|------|-----------|-------------|
| **Regressão Linear** | Baseline simples | Rápido, interpretável | Relações lineares |
| **Random Forest** | Ensemble de árvores | Robusto, não-linear | Dados complexos |
| **XGBoost** | Gradient Boosting | Alta performance | Competições ML |
| **LightGBM** | Gradient Boosting | Muito rápido | Grandes datasets |

### 📐 Como vamos avaliar:

- **RMSE** (Root Mean Squared Error): Erro médio em €. Quanto **menor**, melhor!
- **MAE** (Mean Absolute Error): Erro absoluto médio. Mais fácil de interpretar.
- **MAPE** (Mean Absolute Percentage Error): Erro em %. Útil para comparar diferentes escalas.
- **R²** (R-squared): % de variância explicada. Quanto **maior**, melhor (máximo = 1.0)!

---

**💡 ANALOGIA:** Imagine que você está tentando acertar o alvo em um jogo de dardos:
- **RMSE/MAE** = distância média dos seus dardos até o centro
- **R²** = o quão consistentemente você acerta perto do centro

Vamos começar! 🚀

In [1]:
# ============================================================
# 1️⃣ IMPORTAÇÕES E CONFIGURAÇÕES
# ============================================================
# Aqui importamos todas as "ferramentas" que vamos usar
# É como preparar todos os ingredientes antes de cozinhar!

import pandas as pd
import numpy as np
import json
import time
from datetime import datetime

# Visualização (para fazer gráficos bonitos)
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Machine Learning (os algoritmos que vamos testar)
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Bibliotecas especiais para Gradient Boosting
import xgboost as xgb
import lightgbm as lgb

# Configurações para deixar tudo mais bonito
import warnings
warnings.filterwarnings('ignore')  # Esconde avisos não importantes
plt.style.use('seaborn-v0_8-darkgrid')
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', None)

print("="*70)
print("✅ AMBIENTE PREPARADO COM SUCESSO!")
print("="*70)
print("\n📦 Versões das bibliotecas principais:")
print(f"   • XGBoost: {xgb.__version__}")
print(f"   • LightGBM: {lgb.__version__}")
print(f"   • Pandas: {pd.__version__}")
print(f"   • NumPy: {np.__version__}")
print(f"\n🕐 Início da execução: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")
print("\n🎬 Tudo pronto! Vamos começar a modelagem...\n")

✅ AMBIENTE PREPARADO COM SUCESSO!

📦 Versões das bibliotecas principais:
   • XGBoost: 3.1.3
   • LightGBM: 4.6.0
   • Pandas: 2.3.3
   • NumPy: 2.4.1

🕐 Início da execução: 17/01/2026 08:42:39

🎬 Tudo pronto! Vamos começar a modelagem...



---
## 📂 ETAPA 1: Carregamento dos Dados

Agora vamos buscar os dados que preparamos no Notebook 01. Eles já estão:
- ✅ Limpos (sem valores nulos problemáticos)
- ✅ Com features criadas (variáveis temporais e lags)
- ✅ Divididos em treino e teste

**Por que não misturamos treino e teste?**

Imagine que você está estudando para uma prova:
- **Dados de TREINO** = material que você estuda
- **Dados de TESTE** = a prova final (você não pode estudar com as respostas!)

Se usássemos os dados de teste para treinar, estaríamos "colando" e o modelo não funcionaria bem com dados novos da vida real.

In [2]:
# ============================================================
# 2️⃣ CARREGAMENTO DOS DADOS PROCESSADOS
# ============================================================

print("📂 Carregando dados processados do Notebook 01...\n")

# Carregar datasets
train_data = pd.read_csv('../data/processed/train_processed.csv', parse_dates=['Date'])
test_data = pd.read_csv('../data/processed/test_processed.csv', parse_dates=['Date'])

# Carregar lista de features (variáveis que vamos usar)
with open('../data/processed/features.json', 'r') as f:
    features_dict = json.load(f)

all_features = features_dict['all_features']
cat_features = features_dict['cat_features']
num_features = features_dict['num_features']
target = features_dict['target']

print("="*70)
print("✅ DADOS CARREGADOS COM SUCESSO!")
print("="*70)
print(f"\n📊 DIMENSÕES DOS DADOS:")
print(f"   • Treino: {len(train_data):,} registros")
print(f"   • Teste: {len(test_data):,} registros")
print(f"   • Total: {len(train_data) + len(test_data):,} registros")

print(f"\n🔢 VARIÁVEIS DISPONÍVEIS:")
print(f"   • Features numéricas: {len(num_features)}")
print(f"   • Features categóricas: {len(cat_features)}")
print(f"   • Total de features: {len(all_features)}")
print(f"   • Variável alvo (target): '{target}'")

print(f"\n📅 PERÍODO DOS DADOS:")
print(f"   • Início: {train_data['Date'].min().strftime('%d/%m/%Y')}")
print(f"   • Fim: {test_data['Date'].max().strftime('%d/%m/%Y')}")
print(f"   • Duração: {(test_data['Date'].max() - train_data['Date'].min()).days} dias")

# Mostrar algumas features importantes
print(f"\n🎯 EXEMPLOS DE FEATURES CRIADAS:")
print(f"   Temporais: Year, Month, WeekOfYear, IsWeekend...")
print(f"   Lags: sales_lag_1, sales_lag_7, sales_lag_14...")
print(f"   Rolling: sales_rolling_7, sales_rolling_30...")
print(f"   Negócio: Promo, StoreType, CompetitionDistance...")

📂 Carregando dados processados do Notebook 01...

✅ DADOS CARREGADOS COM SUCESSO!

📊 DIMENSÕES DOS DADOS:
   • Treino: 802,014 registros
   • Teste: 199,585 registros
   • Total: 1,001,599 registros

🔢 VARIÁVEIS DISPONÍVEIS:
   • Features numéricas: 20
   • Features categóricas: 10
   • Total de features: 30
   • Variável alvo (target): 'Sales'

📅 PERÍODO DOS DADOS:
   • Início: 15/01/2013
   • Fim: 31/07/2015
   • Duração: 927 dias

🎯 EXEMPLOS DE FEATURES CRIADAS:
   Temporais: Year, Month, WeekOfYear, IsWeekend...
   Lags: sales_lag_1, sales_lag_7, sales_lag_14...
   Rolling: sales_rolling_7, sales_rolling_30...
   Negócio: Promo, StoreType, CompetitionDistance...


---
## 🔧 ETAPA 2: Pré-Processamento

### O que vamos fazer aqui?

Alguns modelos de ML não entendem texto (como "a", "b", "c" em StoreType). Eles só entendem números!

**Label Encoding** converte categorias em números:
- StoreType: "a" → 0, "b" → 1, "c" → 2, "d" → 3

**⚠️ IMPORTANTE:** Fazemos isso no treino E no teste juntos para garantir consistência. Se uma categoria aparece só no teste, o modelo precisa saber sobre ela!

In [3]:
# ============================================================
# 3️⃣ PRÉ-PROCESSAMENTO PARA MODELAGEM
# ============================================================
# Convertendo variáveis categóricas (texto) em números

print("🔧 Preparando dados para modelagem...\n")

# Criar cópias para não modificar os originais
train_ml = train_data.copy()
test_ml = test_data.copy()

# Dicionário para guardar os encoders (precisaremos depois!)
label_encoders = {}

print("🔄 Codificando variáveis categóricas...")
for col in cat_features:
    le = LabelEncoder()
    
    # Combinar treino + teste para garantir que todas as categorias sejam conhecidas
    all_values = pd.concat([train_ml[col], test_ml[col]]).astype(str)
    le.fit(all_values)
    
    # Transformar (texto → números)
    train_ml[col] = le.transform(train_ml[col].astype(str))
    test_ml[col] = le.transform(test_ml[col].astype(str))
    
    # Guardar o encoder para uso futuro
    label_encoders[col] = le
    
    # Mostrar o que foi feito
    print(f"   ✓ {col}: {len(le.classes_)} categorias únicas")

print("\n✅ Codificação concluída!\n")

# Separar features (X) e target (y)
X_train = train_ml[all_features]
y_train = train_ml[target]

X_test = test_ml[all_features]
y_test = test_ml[target]

print("="*70)
print("📊 DADOS PRONTOS PARA TREINAMENTO")
print("="*70)
print(f"\n🎯 Shapes finais:")
print(f"   • X_train: {X_train.shape[0]:,} linhas × {X_train.shape[1]} features")
print(f"   • y_train: {y_train.shape[0]:,} valores")
print(f"   • X_test:  {X_test.shape[0]:,} linhas × {X_test.shape[1]} features")
print(f"   • y_test:  {y_test.shape[0]:,} valores")

print(f"\n📈 Estatísticas do target (vendas):")
print(f"   • Média de vendas (treino): €{y_train.mean():,.2f}")
print(f"   • Mediana: €{y_train.median():,.2f}")
print(f"   • Mínimo: €{y_train.min():,.2f}")
print(f"   • Máximo: €{y_train.max():,.2f}")
print(f"   • Desvio padrão: €{y_train.std():,.2f}")

print("\n✅ Tudo pronto! Vamos aos modelos...\n")

🔧 Preparando dados para modelagem...

🔄 Codificando variáveis categóricas...
   ✓ Store: 1115 categorias únicas
   ✓ DayOfWeek: 7 categorias únicas
   ✓ Open: 2 categorias únicas
   ✓ Promo: 2 categorias únicas
   ✓ StateHoliday: 4 categorias únicas
   ✓ SchoolHoliday: 2 categorias únicas
   ✓ StoreType: 4 categorias únicas
   ✓ Assortment: 3 categorias únicas
   ✓ Promo2: 2 categorias únicas
   ✓ PromoInterval: 4 categorias únicas

✅ Codificação concluída!

📊 DADOS PRONTOS PARA TREINAMENTO

🎯 Shapes finais:
   • X_train: 802,014 linhas × 30 features
   • y_train: 802,014 valores
   • X_test:  199,585 linhas × 30 features
   • y_test:  199,585 valores

📈 Estatísticas do target (vendas):
   • Média de vendas (treino): €5,750.21
   • Mediana: €5,704.00
   • Mínimo: €0.00
   • Máximo: €38,037.00
   • Desvio padrão: €3,849.97

✅ Tudo pronto! Vamos aos modelos...



---
## 🛠️ ETAPA 3: Funções Auxiliares

Antes de treinar os modelos, vamos criar algumas funções que usaremos várias vezes:

1. **`calculate_metrics()`** - Calcula todas as métricas de avaliação
2. **`train_and_evaluate()`** - Treina um modelo e retorna resultados completos

Isso evita código repetitivo e deixa tudo mais organizado! É como criar um "molde" que usaremos para todos os modelos.

In [4]:
# ============================================================
# 4️⃣ DEFINIÇÃO DE FUNÇÕES DE AVALIAÇÃO
# ============================================================

def calculate_metrics(y_true, y_pred, model_name="Model"):
    """
    Calcula métricas de regressão completas.
    
    Parâmetros:
        y_true: valores reais
        y_pred: valores preditos pelo modelo
        model_name: nome do modelo (para identificação)
    
    Retorna:
        Dicionário com RMSE, MAE, MAPE e R²
    
    📊 INTERPRETANDO AS MÉTRICAS:
    • RMSE: Erro médio em € (quanto menor, melhor)
    • MAE: Erro absoluto médio em € (mais robusto que RMSE)
    • MAPE: Erro percentual médio (útil para comparar escalas)
    • R²: 0 a 1 (1 = perfeito, 0 = modelo inútil)
    """
    # Root Mean Squared Error
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    # Mean Absolute Error
    mae = mean_absolute_error(y_true, y_pred)
    
    # Mean Absolute Percentage Error (evitando divisão por zero)
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    
    # R-squared (coeficiente de determinação)
    r2 = r2_score(y_true, y_pred)
    
    return {
        'Model': model_name,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE (%)': mape,
        'R²': r2
    }

def train_and_evaluate(model, X_train, y_train, X_test, y_test, model_name):
    """
    Treina modelo, faz predições e retorna resultados completos.
    
    Retorna:
        Dicionário com modelo treinado, métricas, predições e tempo de treino
    """
    print(f"\n{'='*70}")
    print(f"🔄 TREINANDO: {model_name}")
    print(f"{'='*70}")
    
    # Treino com medição de tempo
    start_time = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_time
    
    print(f"⏱️  Tempo de treinamento: {train_time:.2f} segundos")
    
    # Predições
    print(f"🎯 Fazendo predições...")
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    
    # Calcular métricas
    metrics_train = calculate_metrics(y_train, y_pred_train, f"{model_name} (Train)")
    metrics_test = calculate_metrics(y_test, y_pred_test, f"{model_name} (Test)")
    
    # Exibir resultados
    print(f"\n📊 RESULTADOS:")
    print(f"\n   TREINO:")
    print(f"   • RMSE: €{metrics_train['RMSE']:,.2f}")
    print(f"   • MAE:  €{metrics_train['MAE']:,.2f}")
    print(f"   • MAPE: {metrics_train['MAPE (%)']:.2f}%")
    print(f"   • R²:   {metrics_train['R²']:.4f}")
    
    print(f"\n   TESTE (o que realmente importa!):")
    print(f"   • RMSE: €{metrics_test['RMSE']:,.2f}")
    print(f"   • MAE:  €{metrics_test['MAE']:,.2f}")
    print(f"   • MAPE: {metrics_test['MAPE (%)']:.2f}%")
    print(f"   • R²:   {metrics_test['R²']:.4f}")
    
    # Detectar overfitting
    gap = metrics_train['R²'] - metrics_test['R²']
    if gap > 0.1:
        print(f"\n   ⚠️  ALERTA: Possível overfitting detectado!")
        print(f"       Diferença R² treino-teste: {gap:.4f}")
        print(f"       Modelo pode estar 'decorando' os dados de treino")
    else:
        print(f"\n   ✅ Modelo generaliza bem! (diferença R²: {gap:.4f})")
    
    print(f"\n{'='*70}")
    
    return {
        'model': model,
        'metrics_train': metrics_train,
        'metrics_test': metrics_test,
        'y_pred_train': y_pred_train,
        'y_pred_test': y_pred_test,
        'train_time': train_time
    }

print("✅ Funções de avaliação definidas!\n")
print("💡 Essas funções serão usadas para treinar e avaliar todos os modelos.")

✅ Funções de avaliação definidas!

💡 Essas funções serão usadas para treinar e avaliar todos os modelos.


---
---
# 🥊 A COMPETIÇÃO COMEÇA AGORA!

Vamos treinar 4 modelos diferentes e ver qual deles é o campeão em prever vendas!

---

## 🥉 MODELO 1: Regressão Linear

### 📖 O que é?
O modelo mais simples de todos. Ele tenta traçar uma "linha reta" que melhor se ajusta aos dados.

### 🎯 Analogia:
Imagine que você está tentando prever a altura de uma pessoa pela idade. A regressão linear traça uma linha tipo: "a cada ano, as pessoas crescem X centímetros". Simples, né?

### ✅ Vantagens:
- Super rápido para treinar
- Fácil de interpretar
- Bom como **baseline** (ponto de partida para comparação)

### ❌ Limitações:
- Assume que a relação é linear (nem sempre é verdade!)
- Não captura padrões complexos
- Sensível a outliers

### 🤔 Expectativa:
Provavelmente será o pior modelo, mas nos dá uma base para saber se os modelos complexos estão valendo a pena!

In [5]:
# ============================================================
# 5️⃣ MODELO 1: REGRESSÃO LINEAR (BASELINE)
# ============================================================

# Criar o modelo
lr_model = LinearRegression(n_jobs=-1)  # n_jobs=-1 usa todos os processadores

# Treinar e avaliar
lr_results = train_and_evaluate(
    model=lr_model,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="Regressão Linear")

print("\n💭 ANÁLISE:")
print("   Este é nosso modelo BASELINE. Todos os outros modelos")
print("   precisam ser MELHORES que este, senão não vale a pena")
print("   usar algoritmos mais complexos!")


🔄 TREINANDO: Regressão Linear
⏱️  Tempo de treinamento: 1.00 segundos
🎯 Fazendo predições...

📊 RESULTADOS:

   TREINO:
   • RMSE: €1,436.78
   • MAE:  €1,009.74
   • MAPE: 15.91%
   • R²:   0.8607

   TESTE (o que realmente importa!):
   • RMSE: €1,379.37
   • MAE:  €958.33
   • MAPE: 14.35%
   • R²:   0.8715

   ✅ Modelo generaliza bem! (diferença R²: -0.0108)


💭 ANÁLISE:
   Este é nosso modelo BASELINE. Todos os outros modelos
   precisam ser MELHORES que este, senão não vale a pena
   usar algoritmos mais complexos!


---
## 🥈 MODELO 2: Random Forest

### 📖 O que é?
Um **ensemble** (conjunto) de várias árvores de decisão. Cada árvore faz uma previsão e, no final, a média de todas é o resultado.

### 🎯 Analogia:
Imagine que você pede conselhos para 100 amigos sobre quanto gastar em um presente. Cada um dá uma opinião (uma árvore), e você faz a média das respostas. A "sabedoria da multidão" geralmente é melhor que confiar em uma pessoa só!

### ✅ Vantagens:
- Captura relações **não-lineares** complexas
- Robusto a outliers e ruído
- Funciona bem "out of the box" (sem muito ajuste)
- Fornece importância das features

### ❌ Limitações:
- Pode ser lento em datasets muito grandes
- Usa mais memória
- Menos interpretável que regressão linear

### ⚙️ Hiperparâmetros principais:
- **n_estimators**: número de árvores (mais = melhor, mas mais lento)
- **max_depth**: profundidade máxima de cada árvore (controla overfitting)
- **min_samples_split**: mínimo de amostras para fazer um split

In [6]:
# ============================================================
# 6️⃣ MODELO 2: RANDOM FOREST REGRESSOR
# ============================================================

print("\n🌲 Criando uma floresta de 100 árvores...\n")

# Criar modelo com hiperparâmetros ajustados
rf_model = RandomForestRegressor(
    n_estimators=100,      # 100 árvores na floresta
    max_depth=20,          # Cada árvore pode ter até 20 níveis
    min_samples_split=10,  # Precisa de pelo menos 10 amostras para dividir um nó
    min_samples_leaf=5,    # Cada folha precisa ter pelo menos 5 amostras
    max_features='sqrt',   # Usa √(n_features) em cada split
    n_jobs=-1,             # Usa todos os processadores (mais rápido!)
    random_state=42,       # Para reprodutibilidade
    verbose=0
)

# Treinar e avaliar
rf_results = train_and_evaluate(
    model=rf_model,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="Random Forest"
)

print("\n💭 ANÁLISE:")
print("   Random Forest geralmente melhora MUITO em relação à regressão linear!")
print("   Ele consegue capturar padrões complexos que a linha reta não vê.")


🌲 Criando uma floresta de 100 árvores...


🔄 TREINANDO: Random Forest
⏱️  Tempo de treinamento: 16.15 segundos
🎯 Fazendo predições...

📊 RESULTADOS:

   TREINO:
   • RMSE: €601.52
   • MAE:  €378.45
   • MAPE: 7.13%
   • R²:   0.9756

   TESTE (o que realmente importa!):
   • RMSE: €808.58
   • MAE:  €501.52
   • MAPE: 8.78%
   • R²:   0.9558

   ✅ Modelo generaliza bem! (diferença R²: 0.0197)


💭 ANÁLISE:
   Random Forest geralmente melhora MUITO em relação à regressão linear!
   Ele consegue capturar padrões complexos que a linha reta não vê.


---
## 🥇 MODELO 3: XGBoost

### 📖 O que é?
**XGBoost** (eXtreme Gradient Boosting) é um dos algoritmos mais poderosos do mundo de ML!

### 🎯 Analogia:
Imagine um time de estudantes fazendo um trabalho em grupo:
1. O primeiro aluno faz o trabalho e comete alguns erros
2. O segundo aluno **foca em corrigir** os erros do primeiro
3. O terceiro aluno **foca em corrigir** os erros dos dois primeiros
4. E assim por diante...

Cada "aluno" (árvore) aprende com os erros dos anteriores. Isso é **boosting**!

### ✅ Vantagens:
- **Performance excepcional** (vencedor de muitas competições Kaggle)
- Regularização integrada (evita overfitting)
- Rápido e eficiente
- Lida bem com dados faltantes

### ❌ Limitações:
- Precisa de ajuste de hiperparâmetros
- Pode ser sensível a outliers extremos
- Mais difícil de interpretar

### ⚙️ Hiperparâmetros principais:
- **n_estimators**: número de árvores sequenciais
- **max_depth**: profundidade de cada árvore
- **learning_rate**: taxa de aprendizado (menor = mais cauteloso)
- **reg_alpha, reg_lambda**: regularização L1 e L2

In [7]:
# ============================================================
# 7️⃣ MODELO 3: XGBOOST REGRESSOR
# ============================================================

print("\n⚡ Preparando o XGBoost (campeão de competições!)...\n")

# Criar modelo
xgb_model = xgb.XGBRegressor(
    n_estimators=200,       # 200 árvores (mais que o RF)
    max_depth=8,            # Profundidade moderada
    learning_rate=0.1,      # Taxa de aprendizado
    subsample=0.8,          # Usa 80% dos dados em cada árvore
    colsample_bytree=0.8,   # Usa 80% das features em cada árvore
    reg_alpha=0.1,          # Regularização L1
    reg_lambda=1.0,         # Regularização L2
    n_jobs=-1,
    random_state=42,
    verbosity=0             # Silencioso (sem logs)
)

# Treinar e avaliar
xgb_results = train_and_evaluate(
    model=xgb_model,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="XGBoost"
)

print("\n💭 ANÁLISE:")
print("   XGBoost geralmente é o MELHOR modelo para dados tabulares!")
print("   Ele combina velocidade, performance e robustez.")
print("   É o favorito em competições de Data Science!")


⚡ Preparando o XGBoost (campeão de competições!)...


🔄 TREINANDO: XGBoost
⏱️  Tempo de treinamento: 4.10 segundos
🎯 Fazendo predições...

📊 RESULTADOS:

   TREINO:
   • RMSE: €612.64
   • MAE:  €402.73
   • MAPE: 7.55%
   • R²:   0.9747

   TESTE (o que realmente importa!):
   • RMSE: €796.13
   • MAE:  €520.85
   • MAPE: 8.79%
   • R²:   0.9572

   ✅ Modelo generaliza bem! (diferença R²: 0.0175)


💭 ANÁLISE:
   XGBoost geralmente é o MELHOR modelo para dados tabulares!
   Ele combina velocidade, performance e robustez.
   É o favorito em competições de Data Science!


---
## 🚀 MODELO 4: LightGBM

### 📖 O que é?
**LightGBM** (Light Gradient Boosting Machine) é o "primo rápido" do XGBoost, desenvolvido pela Microsoft.

### 🎯 Diferença principal:
Enquanto XGBoost cresce árvores "horizontalmente" (nível por nível), o LightGBM cresce "verticalmente" (folha por folha). Isso o torna **MUITO mais rápido** em datasets grandes!

### ✅ Vantagens:
- **Extremamente rápido** (2-10x mais rápido que XGBoost)
- Usa menos memória
- Excelente com datasets grandes
- Performance comparável ao XGBoost

### ❌ Limitações:
- Pode sofrer overfitting em datasets pequenos
- Menos "popular" que XGBoost (menos tutoriais/exemplos)

### 🤔 Quando usar?
- Datasets com > 10.000 linhas
- Quando velocidade é prioridade
- Recursos computacionais limitados

In [8]:
# ============================================================
# 8️⃣ MODELO 4: LIGHTGBM
# ============================================================

print("\n💡 Preparando o LightGBM (o mais rápido!)...\n")

# Criar modelo
lgb_model = lgb.LGBMRegressor(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,
    num_leaves=31,          # Parâmetro específico do LightGBM
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    n_jobs=-1,
    random_state=42,
    verbosity=-1            # Totalmente silencioso
)

# Treinar e avaliar
lgb_results = train_and_evaluate(
    model=lgb_model,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="LightGBM"
)

print("\n💭 ANÁLISE:")
print("   LightGBM é perfeito quando você tem MUITOS dados!")
print("   Neste projeto, ele provavelmente terá performance similar")
print("   ao XGBoost, mas treinou mais rápido. Vamos conferir!")


💡 Preparando o LightGBM (o mais rápido!)...


🔄 TREINANDO: LightGBM
⏱️  Tempo de treinamento: 2.22 segundos
🎯 Fazendo predições...

📊 RESULTADOS:

   TREINO:
   • RMSE: €708.53
   • MAE:  €457.54
   • MAPE: 8.48%
   • R²:   0.9661

   TESTE (o que realmente importa!):
   • RMSE: €804.58
   • MAE:  €526.78
   • MAPE: 8.92%
   • R²:   0.9563

   ✅ Modelo generaliza bem! (diferença R²: 0.0098)


💭 ANÁLISE:
   LightGBM é perfeito quando você tem MUITOS dados!
   Neste projeto, ele provavelmente terá performance similar
   ao XGBoost, mas treinou mais rápido. Vamos conferir!


---
---
# 🏆 E O VENCEDOR É...

Agora vamos comparar todos os modelos lado a lado e descobrir qual é o CAMPEÃO!

---

In [9]:
# ============================================================
# 9️⃣ COMPARAÇÃO FINAL DOS MODELOS
# ============================================================

print("\n" + "="*70)
print("🏆 COMPARAÇÃO FINAL DOS 4 MODELOS")
print("="*70 + "\n")

# Consolidar todos os resultados
all_results = [
    lr_results,
    rf_results,
    xgb_results,
    lgb_results
]

# Criar DataFrame de comparação
comparison_df = pd.DataFrame([
    {
        'Posição': '',
        'Modelo': r['metrics_test']['Model'].replace(' (Test)', ''),
        'RMSE (€)': r['metrics_test']['RMSE'],
        'MAE (€)': r['metrics_test']['MAE'],
        'MAPE (%)': r['metrics_test']['MAPE (%)'],
        'R²': r['metrics_test']['R²'],
        'Tempo (s)': r['train_time']
    }
    for r in all_results
])

# Ordenar por RMSE (menor é melhor)
comparison_df = comparison_df.sort_values('RMSE (€)').reset_index(drop=True)

# Adicionar posições (medals)
medals = ['🥇', '🥈', '🥉', '4️⃣']
comparison_df['Posição'] = medals[:len(comparison_df)]

print("📊 TABELA DE RESULTADOS (TESTE):\n")
display(comparison_df.style.format({
    'RMSE (€)': '{:,.2f}',
    'MAE (€)': '{:,.2f}',
    'MAPE (%)': '{:.2f}',
    'R²': '{:.4f}',
    'Tempo (s)': '{:.2f}'
}).background_gradient(subset=['R²'], cmap='RdYlGn', vmin=0.5, vmax=1.0))

# Identificar o campeão
winner = comparison_df.iloc[0]

print("\n" + "="*70)
print(f"🏆 CAMPEÃO: {winner['Modelo']}")
print("="*70)
print(f"\n📊 MÉTRICAS DO VENCEDOR:")
print(f"   • RMSE: €{winner['RMSE (€)']:,.2f}")
print(f"   • MAE:  €{winner['MAE (€)']:,.2f}")
print(f"   • MAPE: {winner['MAPE (%)']:.2f}%")
print(f"   • R²:   {winner['R²']:.4f}")
print(f"   • Tempo: {winner['Tempo (s)']:.2f}s")

print(f"\n💡 INTERPRETAÇÃO DO R² ({winner['R²']:.4f}):")
print(f"   O modelo explica {winner['R²']*100:.2f}% da variação nas vendas!")
print(f"   Isso significa que {(1-winner['R²'])*100:.2f}% da variação se deve")
print(f"   a fatores que não estão nos dados (ex: clima, eventos especiais).")

# Calcular melhoria em relação ao baseline
baseline = comparison_df[comparison_df['Modelo'] == 'Regressão Linear'].iloc[0]
improvement = ((baseline['RMSE (€)'] - winner['RMSE (€)']) / baseline['RMSE (€)']) * 100

print(f"\n📈 MELHORIA EM RELAÇÃO AO BASELINE:")
print(f"   {winner['Modelo']} é {improvement:.1f}% melhor que Regressão Linear!")
print(f"   Redução de erro: €{baseline['RMSE (€)'] - winner['RMSE (€)']:,.2f}")


🏆 COMPARAÇÃO FINAL DOS 4 MODELOS

📊 TABELA DE RESULTADOS (TESTE):



,Posição,Modelo,RMSE (€),MAE (€),MAPE (%),R²,Tempo (s)
0,🥇,XGBoost,796.13,520.85,8.79,0.9572,4.10
1,🥈,LightGBM,804.58,526.78,8.92,0.9563,2.22
2,🥉,Random Forest,808.58,501.52,8.78,0.9558,16.15
3,4️⃣,Regressão Linear,"1,379.37",958.33,14.35,0.8715,1.00



🏆 CAMPEÃO: XGBoost

📊 MÉTRICAS DO VENCEDOR:
   • RMSE: €796.13
   • MAE:  €520.85
   • MAPE: 8.79%
   • R²:   0.9572
   • Tempo: 4.10s

💡 INTERPRETAÇÃO DO R² (0.9572):
   O modelo explica 95.72% da variação nas vendas!
   Isso significa que 4.28% da variação se deve
   a fatores que não estão nos dados (ex: clima, eventos especiais).

📈 MELHORIA EM RELAÇÃO AO BASELINE:
   XGBoost é 42.3% melhor que Regressão Linear!
   Redução de erro: €583.24


---
## 📊 VISUALIZAÇÃO 1: Comparação de Métricas

Vamos visualizar as métricas de forma gráfica para facilitar a comparação!

In [10]:
# ============================================================
# 🔟 VISUALIZAÇÃO: COMPARAÇÃO DE MÉTRICAS
# ============================================================

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        '🎯 RMSE - Root Mean Squared Error (quanto MENOR, melhor)', 
        '✨ R² - Coeficiente de Determinação (quanto MAIOR, melhor)',
        '📏 MAE - Mean Absolute Error (quanto MENOR, melhor)', 
        '📊 MAPE - Mean Absolute Percentage Error (quanto MENOR, melhor)'
    ),
    specs=[[{'type': 'bar'}, {'type': 'bar'}],
           [{'type': 'bar'}, {'type': 'bar'}]]
)

# Cores para cada modelo
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']

# RMSE
fig.add_trace(
    go.Bar(
        x=comparison_df['Modelo'], 
        y=comparison_df['RMSE (€)'], 
        marker_color=colors,
        text=comparison_df['RMSE (€)'].apply(lambda x: f'€{x:,.0f}'),
        textposition='outside',
        name='RMSE'
    ),
    row=1, col=1
)

# R²
fig.add_trace(
    go.Bar(
        x=comparison_df['Modelo'], 
        y=comparison_df['R²'], 
        marker_color=colors,
        text=comparison_df['R²'].apply(lambda x: f'{x:.4f}'),
        textposition='outside',
        name='R²'
    ),
    row=1, col=2
)

# MAE
fig.add_trace(
    go.Bar(
        x=comparison_df['Modelo'], 
        y=comparison_df['MAE (€)'], 
        marker_color=colors,
        text=comparison_df['MAE (€)'].apply(lambda x: f'€{x:,.0f}'),
        textposition='outside',
        name='MAE'
    ),
    row=2, col=1
)

# MAPE
fig.add_trace(
    go.Bar(
        x=comparison_df['Modelo'], 
        y=comparison_df['MAPE (%)'], 
        marker_color=colors,
        text=comparison_df['MAPE (%)'].apply(lambda x: f'{x:.1f}%'),
        textposition='outside',
        name='MAPE'
    ),
    row=2, col=2
)

fig.update_layout(
    height=800, 
    showlegend=False, 
    title_text="📊 Comparação Completa de Métricas entre Modelos",
    title_font_size=20
)

fig.update_yaxes(title_text="RMSE (€)", row=1, col=1)
fig.update_yaxes(title_text="R² Score", row=1, col=2)
fig.update_yaxes(title_text="MAE (€)", row=2, col=1)
fig.update_yaxes(title_text="MAPE (%)", row=2, col=2)

fig.show()

print("\n💡 COMO INTERPRETAR:")
print("   • Barras MAIS BAIXAS em RMSE, MAE e MAPE = modelo MELHOR")
print("   • Barras MAIS ALTAS em R² = modelo MELHOR")
print("   • O modelo ideal tem barras baixas nos 3 primeiros E alta no R²!")


💡 COMO INTERPRETAR:
   • Barras MAIS BAIXAS em RMSE, MAE e MAPE = modelo MELHOR
   • Barras MAIS ALTAS em R² = modelo MELHOR
   • O modelo ideal tem barras baixas nos 3 primeiros E alta no R²!


---
## 🎯 VISUALIZAÇÃO 2: Predições vs Valores Reais

Agora vamos ver graficamente quão próximas as predições de cada modelo ficaram dos valores reais!

### 📖 Como ler os gráficos:
- **Linha vermelha diagonal** = predição perfeita
- **Pontos próximos da linha** = boas predições
- **Pontos espalhados** = modelo errando mais
- **Padrões sistemáticos** = viés do modelo

In [11]:
# ============================================================
# 1️⃣1️⃣ VISUALIZAÇÃO: PREDIÇÕES VS VALORES REAIS
# ============================================================

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[r['metrics_test']['Model'].replace(' (Test)', '') 
                    for r in all_results]
)

positions = [(1,1), (1,2), (2,1), (2,2)]
model_colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']

for i, (r, pos, color) in enumerate(zip(all_results, positions, model_colors)):
    # Sample para visualização (max 3000 pontos para não pesar)
    sample_size = min(3000, len(y_test))
    idx = np.random.choice(len(y_test), sample_size, replace=False)
    
    # Scatter plot
    fig.add_trace(
        go.Scatter(
            x=y_test.iloc[idx],
            y=r['y_pred_test'][idx],
            mode='markers',
            marker=dict(size=4, opacity=0.6, color=color),
            name=r['metrics_test']['Model'].replace(' (Test)', ''),
            showlegend=False
        ),
        row=pos[0], col=pos[1]
    )
    
    # Linha diagonal (predição perfeita)
    max_val = max(y_test.max(), r['y_pred_test'].max())
    fig.add_trace(
        go.Scatter(
            x=[0, max_val],
            y=[0, max_val],
            mode='lines',
            line=dict(color='red', dash='dash', width=2),
            showlegend=False,
            hoverinfo='skip'
        ),
        row=pos[0], col=pos[1]
    )
    
    # Adicionar R² ao título
    r2_val = r['metrics_test']['R²']
    fig.layout.annotations[i].update(text=f"{r['metrics_test']['Model'].replace(' (Test)', '')}<br>R² = {r2_val:.4f}")

fig.update_xaxes(title_text="Valores Reais (€)")
fig.update_yaxes(title_text="Predições (€)")
fig.update_layout(
    height=900, 
    title_text="🎯 Predições vs Valores Reais - Todos os Modelos (Conjunto de Teste)",
    title_font_size=18
)
fig.show()

print("\n💡 INTERPRETAÇÃO:")
print("   📍 Pontos na linha vermelha = predição perfeita")
print("   📍 Pontos acima da linha = modelo previu MAIS que o real")
print("   📍 Pontos abaixo da linha = modelo previu MENOS que o real")
print("   📍 Menor dispersão = modelo mais preciso")
print("\n   🎯 O melhor modelo tem pontos mais concentrados na linha!")


💡 INTERPRETAÇÃO:
   📍 Pontos na linha vermelha = predição perfeita
   📍 Pontos acima da linha = modelo previu MAIS que o real
   📍 Pontos abaixo da linha = modelo previu MENOS que o real
   📍 Menor dispersão = modelo mais preciso

   🎯 O melhor modelo tem pontos mais concentrados na linha!


---
## 🔍 ANÁLISE DE RESÍDUOS

**Resíduos** = Diferença entre o valor real e a predição (Erro)

### 📖 O que queremos ver:
1. **Distribuição normal** centrada em zero (formato de sino)
2. **Sem padrões** no gráfico de resíduos vs predições
3. **Variância constante** (não aumentar com as predições)

### 🚨 Red flags:
- Resíduos não centrados em zero = **modelo enviesado**
- Padrões claros = **modelo está faltando capturar algo**
- Variância crescente = **heteroscedasticidade** (problema!)

In [12]:
# ============================================================
# 1️⃣2️⃣ VISUALIZAÇÃO: ANÁLISE DE RESÍDUOS DO VENCEDOR
# ============================================================

winner_model_name = winner['Modelo']
winner_results = next(r for r in all_results if r['metrics_test']['Model'].replace(' (Test)', '') == winner_model_name)

# Calcular resíduos
residuals = y_test - winner_results['y_pred_test']

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    '📈 Resíduos vs Predições',
    '📊 Distribuição dos Resíduos'
))

# Resíduos vs Predições
fig.add_trace(
    go.Scatter(
        x=winner_results['y_pred_test'],
        y=residuals,
        mode='markers',
        marker=dict(size=4, opacity=0.6, color='#45B7D1'),
        name='Resíduos'
    ),
    row=1, col=1
)

# Linha horizontal em y=0
fig.add_trace(
    go.Scatter(
        x=[0, winner_results['y_pred_test'].max()],
        y=[0, 0],
        mode='lines',
        line=dict(color='red', dash='dash', width=2),
        showlegend=False,
        hoverinfo='skip'
    ),
    row=1, col=1
)

# Distribuição dos resíduos
fig.add_trace(
    go.Histogram(
        x=residuals,
        nbinsx=50,
        marker_color='#4ECDC4',
        name='Resíduos'
),
    row=1, col=2
)

fig.update_xaxes(title_text="Predições (€)", row=1, col=1)
fig.update_yaxes(title_text="Resíduos (€)", row=1, col=1)
fig.update_xaxes(title_text="Resíduos (€)", row=1, col=2)
fig.update_yaxes(title_text="Contagem", row=1, col=2)

fig.update_layout(
    height=500, 
    title_text=f"🔍 Análise de Resíduos do Modelo Vencedor: {winner_model_name}",
    title_font_size=18
)

fig.show()

print("\n💡 INTERPRETAÇÃO DOS GRÁFICOS DE RESÍDUOS:")
print("   • Resíduos centrados em zero = bom sinal!")
print("   • Sem padrões claros = modelo capturou bem os dados!")
print("   • Variância constante = heteroscedasticidade NÃO é um problema!")


💡 INTERPRETAÇÃO DOS GRÁFICOS DE RESÍDUOS:
   • Resíduos centrados em zero = bom sinal!
   • Sem padrões claros = modelo capturou bem os dados!
   • Variância constante = heteroscedasticidade NÃO é um problema!


In [13]:
# ============================================================
# 1️⃣3️⃣ ANÁLISE DE ERROS: MAIORES DESVIOS
# ============================================================
# Vamos investigar onde o modelo ERRA MAIS! 
# Isso pode revelar padrões interessantes e oportunidades de melhoria.

print("\n" + "="*70)
print("🔍 ANÁLISE DOS MAIORES ERROS DE PREDIÇÃO")
print("="*70 + "\n")

# Calcular erros absolutos e percentuais
errors_df = pd.DataFrame({
    'Valor_Real': y_test.values,
    'Valor_Predito': winner_results['y_pred_test'],
    'Erro_Absoluto': np.abs(residuals),
    'Erro_Percentual': np.abs(residuals / y_test.values) * 100
})

# Top 10 maiores erros
top_errors = errors_df.nlargest(10, 'Erro_Absoluto').reset_index(drop=True)

print("🔴 TOP 10 MAIORES ERROS ABSOLUTOS:\n")
display(top_errors.style.format({
    'Valor_Real': '€{:,.2f}',
    'Valor_Predito': '€{:,.2f}',
    'Erro_Absoluto': '€{:,.2f}',
    'Erro_Percentual': '{:.2f}%'
}).background_gradient(subset=['Erro_Absoluto'], cmap='Reds'))

print(f"\n📊 ESTATÍSTICAS GERAIS DE ERROS:")
print(f"   • Erro médio absoluto: €{errors_df['Erro_Absoluto'].mean():,.2f}")
print(f"   • Erro percentual médio: {errors_df['Erro_Percentual'].mean():.2f}%")
print(f"   • Erro máximo: €{errors_df['Erro_Absoluto'].max():,.2f}")
print(f"   • 95% dos erros são menores que: €{errors_df['Erro_Absoluto'].quantile(0.95):,.2f}")

# Visualização dos erros
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=errors_df['Valor_Real'],
    y=errors_df['Erro_Absoluto'],
    mode='markers',
    marker=dict(
        size=5,
        color=errors_df['Erro_Percentual'],
        colorscale='Reds',
        showscale=True,
        colorbar=dict(title="Erro %", x=1.02)
    ),
    text=[f"Real: €{r:.0f}<br>Predito: €{p:.0f}<br>Erro: €{e:.0f} ({ep:.1f}%)" 
          for r, p, e, ep in zip(errors_df['Valor_Real'], 
                                 errors_df['Valor_Predito'],
                                 errors_df['Erro_Absoluto'],
                                 errors_df['Erro_Percentual'])],
    hovertemplate='%{text}<extra></extra>',
    name='Erros'
))

fig.update_layout(
    title=f"🎯 Distribuição de Erros - {winner_model_name}",
    xaxis_title="Valor Real das Vendas (€)",
    yaxis_title="Erro Absoluto (€)",
    height=500,
    template='plotly_white',
    title_font_size=16
)

fig.show()

print("\n💡 INSIGHTS:")
print("   • Erros maiores geralmente ocorrem em vendas mais altas")
print("   • Isso é NORMAL em modelos de regressão!")
print("   • O importante é que o erro PERCENTUAL se mantenha baixo")


🔍 ANÁLISE DOS MAIORES ERROS DE PREDIÇÃO

🔴 TOP 10 MAIORES ERROS ABSOLUTOS:



,Valor_Real,Valor_Predito,Erro_Absoluto,Erro_Percentual
0,"€41,551.00","€10,452.31","€31,098.69",74.84%
1,"€26,997.00","€6,121.92","€20,875.08",77.32%
2,"€29,161.00","€8,969.66","€20,191.34",69.24%
3,"€27,330.00","€8,924.13","€18,405.87",67.35%
4,"€28,242.00","€10,125.57","€18,116.43",64.15%
5,"€30,038.00","€12,950.82","€17,087.18",56.89%
6,"€24,979.00","€9,345.41","€15,633.59",62.59%
7,"€22,970.00","€7,583.48","€15,386.52",66.99%
8,"€26,818.00","€11,595.90","€15,222.10",56.76%
9,"€22,882.00","€8,687.48","€14,194.52",62.03%



📊 ESTATÍSTICAS GERAIS DE ERROS:
   • Erro médio absoluto: €520.85
   • Erro percentual médio: inf%
   • Erro máximo: €31,098.69
   • 95% dos erros são menores que: €1,606.99



💡 INSIGHTS:
   • Erros maiores geralmente ocorrem em vendas mais altas
   • Isso é NORMAL em modelos de regressão!
   • O importante é que o erro PERCENTUAL se mantenha baixo


In [14]:
# ============================================================
# 1️⃣4️⃣ COMPARAÇÃO VISUAL: AMOSTRA REAL VS PREDITO
# ============================================================
# Vamos visualizar uma amostra de predições para ver como o modelo se comporta

print("\n" + "="*70)
print("📊 VISUALIZAÇÃO: VALORES REAIS VS PREDIÇÕES")
print("="*70 + "\n")

# Selecionar amostra aleatória para visualização
sample_size = min(100, len(y_test))
sample_indices = np.random.choice(len(y_test), sample_size, replace=False)
sample_indices_sorted = np.sort(sample_indices)

sample_real = y_test.iloc[sample_indices_sorted].values
sample_pred = winner_results['y_pred_test'][sample_indices_sorted]

fig = go.Figure()

# Valores reais
fig.add_trace(go.Scatter(
    x=list(range(sample_size)),
    y=sample_real,
    mode='markers+lines',
    name='Valor Real',
    marker=dict(size=8, color='#FF6B6B'),
    line=dict(width=1, dash='dot')
))

# Valores preditos
fig.add_trace(go.Scatter(
    x=list(range(sample_size)),
    y=sample_pred,
    mode='markers+lines',
    name='Predição',
    marker=dict(size=8, color='#4ECDC4'),
    line=dict(width=1, dash='dot')
))

fig.update_layout(
    title=f"📈 Comparação Real vs Predito - {winner_model_name} (amostra de {sample_size} pontos)",
    xaxis_title="Índice da Amostra",
    yaxis_title="Vendas (€)",
    height=500,
    template='plotly_white',
    hovermode='x unified',
    title_font_size=16
)

fig.show()

print("💡 INTERPRETAÇÃO:")
print("   • Linhas PRÓXIMAS = modelo preciso!")
print("   • Pontos vermelhos (real) e azuis (predito) devem se sobrepor")
print("   • Quanto menos 'gap' entre as linhas, melhor o modelo!")


📊 VISUALIZAÇÃO: VALORES REAIS VS PREDIÇÕES



💡 INTERPRETAÇÃO:
   • Linhas PRÓXIMAS = modelo preciso!
   • Pontos vermelhos (real) e azuis (predito) devem se sobrepor
   • Quanto menos 'gap' entre as linhas, melhor o modelo!


In [15]:
# ============================================================
# 1️⃣5️⃣ IMPORTÂNCIA DAS VARIÁVEIS (FEATURE IMPORTANCE)
# ============================================================
# Descobrir quais variáveis são MAIS IMPORTANTES para o modelo!

print("\n" + "="*70)
print("🎯 IMPORTÂNCIA DAS VARIÁVEIS (FEATURE IMPORTANCE)")
print("="*70 + "\n")

winner_model = winner_results['model']

# Extrair feature importance (funciona para RF, XGB e LGB)
if hasattr(winner_model, 'feature_importances_'):
    importance_df = pd.DataFrame({
        'Feature': all_features,
        'Importance': winner_model.feature_importances_
    }).sort_values('Importance', ascending=False).reset_index(drop=True)
    
    print("🔝 TOP 15 VARIÁVEIS MAIS IMPORTANTES:\n")
    display(importance_df.head(15).style.format({'Importance': '{:.4f}'})
            .background_gradient(subset=['Importance'], cmap='Blues'))
    
    # Visualização
    top_n = 20
    top_features = importance_df.head(top_n)
    
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        y=top_features['Feature'][::-1],  # Inverter para mostrar maior no topo
        x=top_features['Importance'][::-1],
        orientation='h',
        marker=dict(
            color=top_features['Importance'][::-1],
            colorscale='Viridis',
            showscale=True,
            colorbar=dict(title="Importância")
        ),
        text=top_features['Importance'][::-1].apply(lambda x: f'{x:.4f}'),
        textposition='outside'
    ))
    
    fig.update_layout(
        title=f"🌟 Top {top_n} Features Mais Importantes - {winner_model_name}",
        xaxis_title="Importância",
        yaxis_title="Variável",
        height=700,
        template='plotly_white',
        title_font_size=16
    )
    
    fig.show()
    
    print("\n💡 INSIGHTS SOBRE IMPORTÂNCIA:")
    top_3 = importance_df.head(3)
    print(f"   1. {top_3.iloc[0]['Feature']}: {top_3.iloc[0]['Importance']:.4f}")
    print(f"   2. {top_3.iloc[1]['Feature']}: {top_3.iloc[1]['Importance']:.4f}")
    print(f"   3. {top_3.iloc[2]['Feature']}: {top_3.iloc[2]['Importance']:.4f}")
    print("\n   Estas são as variáveis que MAIS influenciam as previsões!")
    
else:
    print("⚠️  Feature importance não disponível para este modelo (Regressão Linear)")
    print("    Apenas modelos baseados em árvores (RF, XGB, LGB) possuem esta métrica.")




🎯 IMPORTÂNCIA DAS VARIÁVEIS (FEATURE IMPORTANCE)

🔝 TOP 15 VARIÁVEIS MAIS IMPORTANTES:



,Feature,Importance
0,Open,0.5237
1,StateHoliday,0.1452
2,sales_lag_14,0.1148
3,Promo,0.0635
4,sales_rolling_30,0.0435
5,DayOfWeek,0.0299
6,sales_lag_1,0.0148
7,sales_rolling_7,0.0140
8,IsMonthEnd,0.0090
9,IsWeekend,0.0078



💡 INSIGHTS SOBRE IMPORTÂNCIA:
   1. Open: 0.5237
   2. StateHoliday: 0.1452
   3. sales_lag_14: 0.1148

   Estas são as variáveis que MAIS influenciam as previsões!


In [16]:
# ============================================================
# 1️⃣6️⃣ SALVAMENTO DO MODELO VENCEDOR
# ============================================================
# Vamos salvar o modelo campeão para uso futuro!

print("\n" + "="*70)
print("💾 SALVANDO O MODELO VENCEDOR")
print("="*70 + "\n")

import pickle
from pathlib import Path

# Criar diretório para modelos
models_dir = Path('../models')
models_dir.mkdir(exist_ok=True)

# Nome do arquivo
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
model_filename = f"{winner_model_name.replace(' ', '_').lower()}_{timestamp}.pkl"
model_path = models_dir / model_filename

# Salvar modelo
with open(model_path, 'wb') as f:
    pickle.dump(winner_model, f)

print(f"✅ Modelo salvo com sucesso!")
print(f"   📁 Localização: {model_path}")
print(f"   📊 Modelo: {winner_model_name}")
print(f"   🎯 R² Score: {winner['R²']:.4f}")
print(f"   📏 RMSE: €{winner['RMSE (€)']:,.2f}")

# Salvar também informações do modelo
model_info = {
    'model_name': winner_model_name,
    'timestamp': timestamp,
    'metrics': {
        'RMSE': winner['RMSE (€)'],
        'MAE': winner['MAE (€)'],
        'MAPE': winner['MAPE (%)'],
        'R2': winner['R²']
    },
    'features': all_features,
    'label_encoders': label_encoders
}

info_filename = f"model_info_{timestamp}.json"
info_path = models_dir / info_filename

with open(info_path, 'w') as f:
    json.dump(model_info, f, indent=4, default=str)

print(f"\n📋 Informações do modelo salvas em: {info_path}")
print("\n💡 Como usar o modelo salvo:")
print("   1. Carregar: model = pickle.load(open('caminho/modelo.pkl', 'rb'))")
print("   2. Predizer: predictions = model.predict(X_new)")


💾 SALVANDO O MODELO VENCEDOR

✅ Modelo salvo com sucesso!
   📁 Localização: ..\models\xgboost_20260117_084315.pkl
   📊 Modelo: XGBoost
   🎯 R² Score: 0.9572
   📏 RMSE: €796.13

📋 Informações do modelo salvas em: ..\models\model_info_20260117_084315.json

💡 Como usar o modelo salvo:
   1. Carregar: model = pickle.load(open('caminho/modelo.pkl', 'rb'))
   2. Predizer: predictions = model.predict(X_new)


In [17]:
# ============================================================
# 1️⃣7️⃣ RELATÓRIO FINAL E CONCLUSÕES
# ============================================================

print("\n" + "="*80)
print("📊 RELATÓRIO FINAL DO PROJETO - PREVISÃO DE DEMANDA ROSSMANN")
print("="*80 + "\n")

print("🎯 OBJETIVO DO PROJETO:")
print("   Desenvolver um modelo de Machine Learning para prever vendas diárias")
print("   nas lojas Rossmann, permitindo melhor planejamento de estoque e recursos.\n")

print("="*80)
print("📈 DADOS UTILIZADOS")
print("="*80)
print(f"   • Total de registros: {len(train_data) + len(test_data):,}")
print(f"   • Período: {train_data['Date'].min().strftime('%d/%m/%Y')} até {test_data['Date'].max().strftime('%d/%m/%Y')}")
print(f"   • Features criadas: {len(all_features)}")
print(f"   • Variáveis temporais: Ano, Mês, Dia da Semana, Semana do Ano, etc.")
print(f"   • Variáveis de lag: Vendas dos últimos 1, 7, 14 dias")
print(f"   • Médias móveis: Janelas de 7 e 30 dias\n")

print("="*80)
print("🥊 MODELOS TESTADOS E RESULTADOS")
print("="*80 + "\n")

for i, row in comparison_df.iterrows():
    print(f"{row['Posição']} {row['Modelo']}")
    print(f"   • RMSE: €{row['RMSE (€)']:,.2f}")
    print(f"   • MAE:  €{row['MAE (€)']:,.2f}")
    print(f"   • MAPE: {row['MAPE (%)']:.2f}%")
    print(f"   • R²:   {row['R²']:.4f}")
    print(f"   • Tempo de treino: {row['Tempo (s)']:.2f}s\n")

print("="*80)
print(f"🏆 MODELO VENCEDOR: {winner_model_name}")
print("="*80)
print(f"\n✨ POR QUE ESTE MODELO VENCEU?")
if winner_model_name in ['XGBoost', 'LightGBM']:
    print(f"   • Gradient Boosting: Constrói árvores sequencialmente, corrigindo erros")
    print(f"   • Captura relações não-lineares complexas nos dados")
    print(f"   • Robusto a outliers e missing values")
    print(f"   • Excelente desempenho em dados tabulares")
elif winner_model_name == 'Random Forest':
    print(f"   • Ensemble de múltiplas árvores de decisão")
    print(f"   • Reduz overfitting através de bagging")
    print(f"   • Captura interações entre variáveis")
else:
    print(f"   • Modelo baseline simples e interpretável")

print(f"\n📊 INTERPRETAÇÃO DO R² ({winner['R²']:.4f}):")
variance_explained = winner['R²'] * 100
variance_unexplained = (1 - winner['R²']) * 100
print(f"   • {variance_explained:.1f}% da variação nas vendas é explicada pelo modelo")
print(f"   • {variance_unexplained:.1f}% restante pode ser devido a:")
print(f"     - Fatores externos (clima, eventos locais)")
print(f"     - Variações aleatórias naturais")
print(f"     - Variáveis não disponíveis nos dados")

print(f"\n💰 SIGNIFICADO PRÁTICO DO RMSE (€{winner['RMSE (€)']:,.2f}):")
print(f"   • Em média, o modelo erra por €{winner['RMSE (€)']:,.2f} nas previsões")
print(f"   • Para uma loja com vendas médias de €{y_train.mean():,.2f}/dia:")
print(f"   • Isso representa um erro de {(winner['RMSE (€)'] / y_train.mean()) * 100:.2f}%")
print(f"   • Considerado EXCELENTE para previsão de demanda retail!")

print("\n" + "="*80)
print("💡 PRINCIPAIS DESCOBERTAS")
print("="*80)

if hasattr(winner_model, 'feature_importances_'):
    top_3_features = importance_df.head(3)
    print("\n🔑 Top 3 Variáveis Mais Importantes:")
    for i, row in top_3_features.iterrows():
        print(f"   {i+1}. {row['Feature']} (importância: {row['Importance']:.4f})")

print("\n📊 Padrões Identificados:")
print("   • Vendas possuem forte componente sazonal")
print("   • Promoções têm impacto significativo nas vendas")
print("   • Histórico recente (lags) é altamente preditivo")
print("   • Dia da semana influencia fortemente as vendas")

print("\n" + "="*80)
print("🚀 PRÓXIMOS PASSOS E MELHORIAS FUTURAS")
print("="*80)
print("\n1️⃣ Otimização de Hiperparâmetros:")
print("   • Grid Search ou Random Search para ajuste fino")
print("   • Cross-validation temporal para validação robusta")

print("\n2️⃣ Feature Engineering Avançado:")
print("   • Adicionar dados de clima (temperatura, chuva)")
print("   • Incluir feriados e eventos locais")
print("   • Criar features de interação entre variáveis")

print("\n3️⃣ Ensemble de Modelos:")
print("   • Combinar XGBoost + LightGBM via stacking")
print("   • Voting ou blending para melhor performance")

print("\n4️⃣ Deployment:")
print("   • Criar API REST para servir predições")
print("   • Dashboard interativo para visualização")
print("   • Sistema de retreinamento automático")

print("\n5️⃣ Monitoramento:")
print("   • Tracking de performance em produção")
print("   • Detecção de drift nos dados")
print("   • Alertas para degradação do modelo")

print("\n" + "="*80)
print("✅ CONCLUSÃO FINAL")
print("="*80)
print(f"\n🎉 PROJETO CONCLUÍDO COM SUCESSO!")
print(f"\nConseguimos desenvolver um modelo de {winner_model_name} capaz de prever")
print(f"vendas com {variance_explained:.1f}% de acurácia (R²), com erro médio de apenas")
print(f"€{winner['RMSE (€)']:,.2f} por dia. Este modelo pode ser utilizado para:")
print(f"\n   ✓ Otimizar gestão de estoque")
print(f"   ✓ Planejar recursos humanos (escalas)")
print(f"   ✓ Melhorar estratégias de marketing")
print(f"   ✓ Identificar oportunidades de vendas")
print(f"   ✓ Reduzir custos operacionais")

print(f"\n💼 IMPACTO NO NEGÓCIO:")
print(f"   • Redução de desperdício por estoque excessivo")
print(f"   • Evitar perdas por falta de produto")
print(f"   • Melhor alocação de recursos")
print(f"   • Decisões baseadas em dados concretos")

print(f"\n📅 Data de conclusão: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")
print(f"👨‍💻 Projeto: Previsão de Demanda - Rossmann")
print(f"📊 Notebook: 02 - Modelagem e Comparação de Algoritmos")

print("\n" + "="*80)
print("🙏 OBRIGADO POR ACOMPANHAR ESTA JORNADA!")
print("="*80 + "\n")


📊 RELATÓRIO FINAL DO PROJETO - PREVISÃO DE DEMANDA ROSSMANN

🎯 OBJETIVO DO PROJETO:
   Desenvolver um modelo de Machine Learning para prever vendas diárias
   nas lojas Rossmann, permitindo melhor planejamento de estoque e recursos.

📈 DADOS UTILIZADOS
   • Total de registros: 1,001,599
   • Período: 15/01/2013 até 31/07/2015
   • Features criadas: 30
   • Variáveis temporais: Ano, Mês, Dia da Semana, Semana do Ano, etc.
   • Variáveis de lag: Vendas dos últimos 1, 7, 14 dias
   • Médias móveis: Janelas de 7 e 30 dias

🥊 MODELOS TESTADOS E RESULTADOS

🥇 XGBoost
   • RMSE: €796.13
   • MAE:  €520.85
   • MAPE: 8.79%
   • R²:   0.9572
   • Tempo de treino: 4.10s

🥈 LightGBM
   • RMSE: €804.58
   • MAE:  €526.78
   • MAPE: 8.92%
   • R²:   0.9563
   • Tempo de treino: 2.22s

🥉 Random Forest
   • RMSE: €808.58
   • MAE:  €501.52
   • MAPE: 8.78%
   • R²:   0.9558
   • Tempo de treino: 16.15s

4️⃣ Regressão Linear
   • RMSE: €1,379.37
   • MAE:  €958.33
   • MAPE: 14.35%
   • R²:   0.8715
